# Análisis de reseñas con NLP — Sector Banca
**Autor:** Alejandro Pujana Quintero · MSc Data Science & IA, Evolve Academy  
**Entorno:** Python 3.12 · Mac Intel · sin GPU · sin servicios de pago

**Hilo del análisis:** sesgo de medición (F1) → empresa + competencia (F2) → limpiar texto (F3) → tono por reseña (F4) → temas (F5) → cruce tema × tono (F6) → visualizar (F7) → recomendar (F8)

---
## Fase 0 — Entorno e imports

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Rutas del proyecto
ROOT = Path("..")
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"

# Semilla global para reproducibilidad
SEED = 42

print("Entorno listo.")

---
## Fase 1 — EDA y demostración del sesgo de estrellas

**Qué hacemos:** cargamos las 123k reseñas y demostramos con números que las estrellas están manipuladas. No es un trámite: es el primer hallazgo y el gancho de la presentación.

**Argumento clave:** en el catálogo de 558 empresas con 100 reseñas, todas tienen media 3.00 y desviación estándar 1.421 — exactamente la cifra que sale si cada empresa tiene 20 reseñas de cada nivel (1★ a 5★). Distribución forzada artificialmente. **Implicación:** el nivel de estrellas no distingue bancos; trabajamos con el texto y con diferencias contra el sector, no con niveles absolutos.

In [ ]:
# --- Carga ---
# df_raw = pd.read_csv(DATA_DIR / "nombre_archivo.csv")
# print(df_raw.shape)
# df_raw.head()
pass

In [ ]:
# --- Demostración del sesgo ---
# stats_por_empresa = df_raw.groupby("company")["stars"].agg(["mean", "std", "count"])
# empresas_100 = stats_por_empresa[stats_por_empresa["count"] == 100]
# print(empresas_100[["mean", "std"]].describe())
pass

In [ ]:
# --- Gráfico: reparto plano de estrellas ---
# (una empresa de ejemplo para ilustrar el amaño)
pass

**Conclusión F1:** las estrellas tienen distribución forzada; no las usamos como variable de resultado. El sentimiento se mide desde el texto (Fase 4). Las comparaciones serán diferencias contra el sector (mismo sesgo en ambos lados → diferencia limpia).

---
## Fase 2 — Empresa objetivo y pool de competencia

**Qué hacemos:** elegir el target entre 2-3 finalistas con 6 criterios basados en datos, y construir el pool (~12-15 empresas) por similitud coseno sobre embeddings de la columna `description`.

**Orden de filtros obligatorio:** primero reducir a empresas de exactamente 100 reseñas (mismo amaño), *luego* medir similitud coseno. Invertir el orden reintroduce el sesgo de la Fase 1.

In [ ]:
# --- Filtro: empresas con exactamente 100 reseñas ---
# df_100 = ...
pass

In [ ]:
# --- Embeddings de descriptions + similitud coseno ---
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# model_emb = SentenceTransformer("all-MiniLM-L6-v2")
# ...
pass

In [ ]:
# --- Scoring de finalistas (6 criterios) + elección del target ---
# TARGET = "..."
# POOL = [...]
pass

**Conclusión F2:** target = `[PENDIENTE]`. Pool = `[PENDIENTE]` empresas. A partir de aquí trabajamos solo con este subconjunto (~1.300-1.600 reseñas).

---
## Fase 3 — Exploración y limpieza del texto

**Qué hacemos:** dos limpiezas distintas porque el texto alimenta dos modelos con necesidades opuestas.

| | Limpieza ligera (sentimiento) | Limpieza pesada (temas) |
|---|---|---|
| Mayúsculas, signos, emojis | se conservan | se quitan |
| Negaciones ("no", "never") | **se conservan** | se quitan |
| Stop words ("el", "the") | se conservan | **se quitan** |
| HTML y URLs | se quitan | se quitan |

In [ ]:
import re

def limpiar_ligero(texto: str) -> str:
    """
    Limpieza mínima para el modelo de sentimiento.

    Quita solo HTML y URLs; conserva mayúsculas, signos, emojis y negaciones
    porque el tono depende de ellos.

    Parameters
    ----------
    texto : str

    Returns
    -------
    str
    """
    texto = re.sub(r"<[^>]+>", " ", texto)          # HTML
    texto = re.sub(r"http\S+|www\.\S+", " ", texto) # URLs
    return texto.strip()


def limpiar_pesado(texto: str, stop_words: set) -> str:
    """
    Limpieza agresiva para topic modeling.

    Quita HTML, URLs, signos, números, mayúsculas y stop words.
    Lo que queda son las palabras con contenido semántico.

    Parameters
    ----------
    texto : str
    stop_words : set

    Returns
    -------
    str
    """
    texto = re.sub(r"<[^>]+>", " ", texto)
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)
    texto = texto.lower()
    texto = re.sub(r"[^a-z\s]", " ", texto)
    tokens = [t for t in texto.split() if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

In [ ]:
# --- Exploración: longitud de reseñas, idiomas, caracteres raros ---
pass

In [ ]:
# --- Aplicar las dos limpiezas al subconjunto target + pool ---
# df_sub["texto_ligero"] = df_sub["review"].apply(limpiar_ligero)
# df_sub["texto_pesado"] = df_sub["review"].apply(lambda x: limpiar_pesado(x, STOP_WORDS))
pass

---
## Fase 4 — Sentimiento (¿hablan bien o mal?)

**Qué hacemos:** asignar nota 1-5 a cada reseña con `nlptown/bert-base-multilingual-uncased-sentiment`, un transformer ya entrenado. No entrenamos nada: tomamos prestado un modelo que ya hizo ese trabajo en múltiples idiomas.

- Aplicamos sobre `texto_ligero` (negaciones y signos intactos).
- Solo sobre target + pool (~1.500 reseñas) — no las 123k.
- **Negativa = nota 1 o 2.** El 3 es ambiguo y lo dejamos aparte.
- Guardar resultado en disco para no recalcular en cada ejecución.

In [ ]:
# from transformers import pipeline

# SENTIMENT_CACHE = TAB_DIR / "sentimiento.parquet"

# if SENTIMENT_CACHE.exists():
#     df_sent = pd.read_parquet(SENTIMENT_CACHE)
# else:
#     pipe = pipeline(
#         "text-classification",
#         model="nlptown/bert-base-multilingual-uncased-sentiment",
#         truncation=True,
#         max_length=512,
#     )
#     # procesar en batches, ordenar por longitud antes para eficiencia
#     # ...
#     df_sent.to_parquet(SENTIMENT_CACHE, index=False)
pass

In [ ]:
# --- Reseñas truncadas (nota al pie, no problema) ---
# n_truncadas = ...
# print(f"Reseñas truncadas (>512 tokens): {n_truncadas} ({n_truncadas/len(df_sub)*100:.1f}%)")
pass

---
## Fase 5 — Temas (¿de qué hablan?)

**Qué hacemos:** descubrir de qué hablan las reseñas con TF-IDF + NMF. Un único modelo sobre target + pool juntos — condición necesaria para que los temas sean comparables entre tu banco y la competencia.

- **TF-IDF:** pondera palabras características de cada reseña (frecuentes en ella, raras en el corpus).
- **NMF:** factoriza la matriz TF-IDF en grupos de palabras que co-ocurren → temas.
- Número de temas: probar 6-12 y quedarse con los que se puedan nombrar en lenguaje de negocio.
- Contrastar con BERTopic y elegir el que dé temas más interpretables.

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.decomposition import NMF

# --- Quitar duplicados antes de TF-IDF ---
# df_sub = df_sub.drop_duplicates(subset="texto_pesado")

# --- Vectorizar ---
# vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
# X_tfidf = vectorizer.fit_transform(df_sub["texto_pesado"])

# --- NMF con N temas ---
# N_TOPICS = 8  # ajustar tras explorar 6-12
# nmf = NMF(n_components=N_TOPICS, random_state=SEED)
# W = nmf.fit_transform(X_tfidf)  # reseñas × temas
# df_sub["tema"] = W.argmax(axis=1)
pass

In [ ]:
# --- Top palabras por tema + asignación de nombre de negocio ---
# NOMBRES_TEMAS = {
#     0: "Reembolsos y disputas",
#     1: "App y banca digital",
#     # ...
# }
pass

---
## Fase 6 — Cruce tema × sentimiento

**Qué hacemos:** el corazón del proyecto. Para cada tema calculamos volumen (% de reseñas) y negatividad (% negativas), para target y para el pool por separado. La señal es la **diferencia** target − pool: positiva = peor que el sector en ese tema.

**Prioridad:** `negatividad × volumen × diferencia_vs_sector`. Un tema con negatividad moderada pero alto volumen supera a uno muy negativo con pocos menciones.

In [ ]:
# --- Tabla tema × {volumen, negatividad} para target y pool ---
# def resumen_por_tema(df, nombre):
#     ...

# tabla_target = resumen_por_tema(df_sub[df_sub["empresa"] == TARGET], "target")
# tabla_pool   = resumen_por_tema(df_sub[df_sub["empresa"].isin(POOL)], "pool")
# tabla_final  = tabla_target.merge(tabla_pool, on="tema")
# tabla_final["diff_negatividad"] = tabla_final["neg_target"] - tabla_final["neg_pool"]
pass

In [ ]:
# --- Top 3 temas prioritarios + reseñas reales ---
# Para el tema más urgente: las 5-10 reseñas más negativas del target
pass

---
## Fase 7 — Visualizaciones para audiencia no técnica

Regla: títulos en lenguaje de negocio, ejes etiquetados, frase de interpretación bajo cada gráfico. Sin "Dimensión 1".

1. Reparto de tono — target vs. pool (barras)
2. Volumen por tema — target vs. pool (barras horizontales)
3. Mapa de calor tema × negatividad (target | pool lado a lado)
4. Cuadrante de prioridad (volumen vs. negatividad, color = diferencia vs. sector)

In [ ]:
# --- Gráfico 1: reparto de tono ---
pass

In [ ]:
# --- Gráfico 2: volumen por tema ---
pass

In [ ]:
# --- Gráfico 3: mapa de calor tema × negatividad ---
pass

In [ ]:
# --- Gráfico 4: cuadrante de prioridad ---
pass

---
## Fase 8 — Conclusiones y recomendaciones

Criterio de apropiación: si la recomendación se puede aplicar a cualquier empresa del mundo, no sirve. Tiene que salir de *estos* datos y *estas* reseñas.

### Contraste de hipótesis

| Hipótesis | Resultado | Evidencia |
|-----------|-----------|----------|
| H1: ... | Confirmada / Rechazada | ... |
| H2: ... | Confirmada / Rechazada | ... |

### Recomendaciones priorizadas

1. **[Tema más urgente]:** acción concreta derivada de las reseñas reales (F6). *Evidencia: "[cita de reseña]".*
2. ...

### Limitaciones

- El sesgo de estrellas obliga a trabajar con diferencias; el nivel absoluto de negatividad no es interpretable.
- Reseñas con < 10 observaciones en un tema → estimaciones inestables, no presentar como hallazgo firme.
- Reseñas en otros idiomas: `langdetect` filtra, pero puede haber errores de clasificación.